# 01 – Raw Data: Dataset Original

**Proyecto:** Predicción de subempleo por insuficiencia de horas en trabajadores ocupados de Lima Metropolitana  
**Objetivo del modelo:** Predecir la probabilidad de que un trabajador ocupado presente subempleo por insuficiencia de horas usando variables sociodemográficas, educativas, laborales, de ingresos y protección social.  
**Target:** `P209H` recodificada como binaria → `1` = tiene voluntad y disponibilidad de trabajar más horas · `0` = no la tiene

## 1. Descripción de la fuente de datos

- **Fuente:** Encuesta Permanente de Empleo Nacional (EPEN) 2024
- **Institución:** Instituto Nacional de Estadística e Informática (INEI) – Perú
- **Archivo:** `250806 EPEN2024.xlsx`
- **Período de referencia:** Trimestre Sep-Oct-Nov 2024
- **Unidad de análisis:** Personas residentes de **14 años a más, ocupadas, en Lima Metropolitana** (`REGION == 1`)
- **Cobertura geográfica:** Nacional (filtrado a Lima Metropolitana)

---

## 2. Variable objetivo (target)

| Variable | Pregunta original | Recodificación |
|---|---|---|
| `P209H` | ¿TUVO LA VOLUNTAD DE TRABAJAR MÁS HORAS Y ADEMÁS ESTUVO DISPONIBLE PARA HACERLO? | `1` = Sí (subempleado por horas) · `0` = No |

> Corresponde a la combinación de `C333` (quería trabajar más horas) y `C334` (estuvo disponible para hacerlo).

---

## 3. Variables del dataset (132 columnas según diccionario INEI)

### Identificación y diseño muestral
| Variable | Descripción |
|---|---|
| `ANIO` | Año de la encuesta |
| `MES` | Mes de la encuesta |
| `CONGLOMERADO` | Número del conglomerado |
| `MUESTRA` | N° de sub muestra |
| `REGION` | Región (1=Lima Met. · 2=Resto urbano · 3=Rural) |
| `ESTRATO` | Estrato geográfico (1–8) |
| `LLAVE_PANEL` | Código de persona panel |

### Características sociodemográficas
| Variable | Descripción |
|---|---|
| `C201` | N° de orden / código de persona |
| `C203` | Relación de parentesco con el jefe del hogar (1=Jefe … 11=Otro no pariente) |
| `C207` | Sexo (1=Hombre · 2=Mujer) |
| `C208` | Edad en años cumplidos |
| `C301_DIA/MES/ANIO` | Fecha de nacimiento |

### Condición de actividad (módulo 300)
| Variable | Descripción |
|---|---|
| `C303` | La semana pasada, ¿tuvo algún trabajo? (1=Sí · 2=No) |
| `C304` | ¿Tiene empleo fijo al que próximamente volverá? |
| `C305` | ¿Tiene negocio propio al que próximamente volverá? |
| `C306_1…C306_11` | Actividades realizadas ≥1 hora para obtener ingresos |

### Ocupación principal
| Variable | Descripción |
|---|---|
| `C308_COD` | Código de ocupación principal (CNO) |
| `C309_COD` | Código de actividad económica de la empresa (CIIU) |
| `C310` | Categoría ocupacional (1=Empleador · 2=Independiente · 3=Empleado/obrero · 4–10=otras) |
| `C311` | Tipo de empleador (1=FF.AA./PNP · 2=Admin. pública · 3=Empresa pública · 4=Service · 5=Empresa privada · 6=Otra) |
| `C312` | Registro en SUNAT (1=Persona jurídica · 2=Natural con RUC · 3=No registrado · 4=No sabe) |
| `C313` | Lleva libros contables (1=Sí · 2=No · 3=No sabe) |
| `C317` | Tamaño de empresa (1=≤20 · 2=21–50 · …) |

### Horas trabajadas
| Variable | Descripción |
|---|---|
| `C318_1…C318_7` | Horas trabajadas cada día (lunes a sábado) en ocupación principal |
| `C318_T` | Total horas trabajadas en ocupación principal |
| `C328_T` | Horas trabajadas en ocupaciones secundarias |
| `whoraT` | Horas totales (todas las ocupaciones) |
| `C330` | ¿Normalmente trabaja esas horas? (1=Sí · 2=No) |
| `C331` | Horas normales semanales en todas las ocupaciones |

### Subempleo por horas (origen del target)
| Variable | Descripción |
|---|---|
| `C333` | ¿La semana pasada quería trabajar más horas? (1=Sí · 2=No) |
| `C334` | ¿Estuvo disponible para trabajar más horas? (1=Sí · 2=No) |
| **`P209H`** | **¿Tuvo voluntad Y disponibilidad de trabajar más horas? (TARGET)** |

### Protección social
| Variable | Descripción |
|---|---|
| `SEGURO1` | Indicador de afiliación a algún seguro de salud |
| `C361_1…C361_8` | Tipos de seguro de salud (SIS, EsSalud, privado, FFAA, etc.) |
| `C364_1…C364_4` | Tipos de sistema de pensiones (AFP, SNP, etc.) |

### Ingresos
| Variable | Descripción |
|---|---|
| `C339_1` | Ingreso en ocupación principal (sin descuentos) |
| `C341_T` | Total ingresos ocupación principal |
| `INGTOT` | Ingreso total mensual |
| `INGTOTP` | Ingreso total per cápita |
| `ingtrabw` | Ingreso laboral (winsorizado) |

### Factores de expansión y otros
| Variable | Descripción |
|---|---|
| `OCUP300` | Ocupación principal según clasificador |
| `RESIDENT` | Condición de residencia |
| `fa_ond24 / fa_efm24 / fa_amj24 / fa_jas24` | Factor de expansión por trimestre |

In [2]:
# ─── Librerías ────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import os

In [3]:
# ─── Rutas ────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = os.getcwd()                                          # .../02_data_understanding/
PROJECT_DIR  = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))   # .../ml_project/
DATA_PATH    = os.path.join(PROJECT_DIR, '01_raw_data', '250806 EPEN2024.xlsx')

print(f'Archivo: {DATA_PATH}')
print(f'Existe:  {os.path.exists(DATA_PATH)}')

# ─── Carga del dataset completo (raw, sin modificaciones) ─────────────────────
df_raw = pd.read_excel(DATA_PATH)

print(f'\nDataset completo cargado: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas')
df_raw.head()

Archivo: g:\Mi unidad\UP - Ingeniería de la información\Semestre IX\Machine Learning\TrabajoFinal\clonProyecto\ML_PROYECTO_26_1\ml_project\01_raw_data\250806 EPEN2024.xlsx
Existe:  True

Dataset completo cargado: 52,251 filas × 132 columnas


,ANIO,MES,CONGLOMERADO,MUESTRA,SELVIV,HOGAR,REGION,LLAVE_PANEL,ESTRATO,C201,...,D350,D351_T,INGTOT,INGTOTP,ingtrabw,RESIDENT,fa_ond24,fa_efm24,fa_amj24,fa_jas24
0,2024,10,18117,2,61,1,1,2.024102e+17,1,1,...,,,1367,1367,1367,1,700.400712,NaN,NaN,NaN
1,2024,10,18117,2,61,1,1,2.024102e+17,1,2,...,,,1534,1534,1534,1,700.400712,NaN,NaN,NaN
2,2024,10,18117,2,61,1,1,2.024102e+17,1,3,...,,,,,,1,,NaN,NaN,NaN
3,2024,10,1823302,1,59,1,1,2.024102e+19,1,1,...,,,1400,1400,1400,1,896.469464,NaN,NaN,NaN
4,2024,10,1823302,1,59,1,1,2.024102e+19,1,2,...,,,1516,1516,1516,1,844.03608,NaN,NaN,NaN


## 4. Universo analítico (solo referencia – sin modificar raw)

Los filtros que se aplicarán en el paso de preprocesamiento son:

1. `REGION == 1` → Lima Metropolitana  
2. `C208 >= 14` → Personas de 14 años a más  
3. `RESIDENT == 1` → Residentes habituales del hogar  
4. Condición de ocupado (definida a partir de `C303`, `C304`, `C305`, `C306_*`)  
5. `P209H` no nulo → tiene información sobre el target  

A continuación se muestra un conteo preliminar **sin aplicar filtros** para entender la distribución.

In [4]:
# ─── Distribución preliminar por REGION ──────────────────────────────────────
print('=== Distribución por REGION ===')
print(df_raw['REGION'].value_counts().rename({1: 'Lima Metropolitana', 2: 'Resto urbano', 3: 'Rural'}))

# C208 puede venir como object desde el Excel → convertir a numérico
edad_num = pd.to_numeric(df_raw['C208'], errors='coerce')
print('\n=== Rango de edad (C208) ===')
print(edad_num.describe())

# P209H tiene espacios/cadenas vacías como missing → limpiar antes de contar
p209h_clean = df_raw['P209H'].replace(r'^\s*$', np.nan, regex=True)
print('\n=== Target P209H (dataset completo) ===')
print(p209h_clean.value_counts(dropna=False))

# Conteo anticipado del universo analítico
mask_lima   = df_raw['REGION'] == 1
mask_edad   = edad_num >= 14
mask_target = p209h_clean.notna()
n_universo  = (mask_lima & mask_edad & mask_target).sum()
print(f'\nRegistros que cumplen REGION=1 + edad≥14 + P209H no nulo: {n_universo:,}')

=== Distribución por REGION ===
REGION
Lima Metropolitana    52251
Name: count, dtype: int64

=== Rango de edad (C208) ===
count    51927.000000
mean        37.712231
std         21.925085
min          1.000000
25%         19.000000
50%         36.000000
75%         54.000000
max        105.000000
Name: C208, dtype: float64

=== Target P209H (dataset completo) ===
P209H
NaN    28197
2.0    18064
1.0     5990
Name: count, dtype: int64

Registros que cumplen REGION=1 + edad≥14 + P209H no nulo: 24,054


C:\Users\guill\AppData\Local\Temp\ipykernel_12396\1317391521.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  p209h_clean = df_raw['P209H'].replace(r'^\s*$', np.nan, regex=True)


In [5]:
# ─── Información general ──────────────────────────────────────────────────────
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52251 entries, 0 to 52250
Columns: 132 entries, ANIO to fa_jas24
dtypes: float64(2), int64(12), object(118)
memory usage: 52.6+ MB


In [6]:
# ─── Estadísticas descriptivas ────────────────────────────────────────────────
df_raw.describe(include='all')

,ANIO,MES,CONGLOMERADO,MUESTRA,SELVIV,HOGAR,REGION,LLAVE_PANEL,ESTRATO,C201,...,D350,D351_T,INGTOT,INGTOTP,ingtrabw,RESIDENT,fa_ond24,fa_efm24,fa_amj24,fa_jas24
count,52251.0,52251.000000,5.225100e+04,52251.000000,52251.000000,52251.000000,52251.0,5.225100e+04,52251.0,52251.000000,...,52251,52251,52251,52251,52251,52251.000000,13673,14474,14147,9957.000000
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,...,64,926,3022,2844,4119,NaN,4988,5202,5143,NaN
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,...,,,,,,NaN,,,,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51165.0,NaN,...,51967,44663,28971,28971,28971,NaN,4144,4273,3983,NaN
mean,2024.0,6.302195,7.480501e+05,1.001569,61.438135,1.028095,1.0,6.155953e+18,NaN,2.497139,...,NaN,NaN,NaN,NaN,NaN,0.978565,NaN,NaN,NaN,891.963443
std,0.0,3.550091,1.202647e+06,0.039584,53.073330,0.195194,0.0,1.006816e+19,NaN,1.537576,...,NaN,NaN,NaN,NaN,NaN,0.144831,NaN,NaN,NaN,330.102831
min,2024.0,1.000000,1.726000e+04,1.000000,1.000000,1.000000,1.0,2.022102e+17,NaN,1.000000,...,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,155.548416
25%,2024.0,3.000000,2.113600e+04,1.000000,26.000000,1.000000,1.0,2.023082e+17,NaN,1.000000,...,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,701.882083
50%,2024.0,6.000000,2.752600e+04,1.000000,52.000000,1.000000,1.0,2.024042e+17,NaN,2.000000,...,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,902.413456
75%,2024.0,10.000000,1.835501e+06,1.000000,84.000000,1.000000,1.0,2.023042e+19,NaN,3.000000,...,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,1084.029270


In [7]:
# ─── Guardar snapshot del raw data (sin ninguna modificación) ─────────────────
SNAPSHOT_DIR  = os.path.join(PROJECT_DIR, 'data', 'raw')
os.makedirs(SNAPSHOT_DIR, exist_ok=True)
SNAPSHOT_PATH = os.path.join(SNAPSHOT_DIR, 'epen2024_raw_snapshot.csv')
df_raw.to_csv(SNAPSHOT_PATH, index=False, encoding='utf-8-sig')
print(f'Snapshot raw guardado: {os.path.abspath(SNAPSHOT_PATH)}')
print(f'Filas: {df_raw.shape[0]:,}  |  Columnas: {df_raw.shape[1]}')

Snapshot raw guardado: g:\Mi unidad\UP - Ingeniería de la información\Semestre IX\Machine Learning\TrabajoFinal\clonProyecto\ML_PROYECTO_26_1\ml_project\data\raw\epen2024_raw_snapshot.csv
Filas: 52,251  |  Columnas: 132
